In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
train = pd.read_csv('/content/drive/MyDrive/train_sam.csv')
test = pd.read_csv('/content/drive/MyDrive/test_sam.csv')

### 0) 기본 세팅 + train/valid split

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

TARGET = "target"

# X/y 분리
X = train.drop(columns=[TARGET])
y = train[TARGET].astype(int)  # 보통 0/1

# train/valid split
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train:", X_tr.shape, "valid:", X_val.shape, "test:", test.shape)
print("pos rate:", y.mean(), "val pos rate:", y_val.mean())

### 1) Baseline: Logistic Regression

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 컬럼 타입 자동 분류
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

print("num_cols:", len(num_cols), "cat_cols:", len(cat_cols))

numeric_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_tf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_tf, num_cols),
        ("cat", categorical_tf, cat_cols),
    ],
    remainder="drop"
)

logit = LogisticRegression(
    max_iter=2000,
    n_jobs=-1,
    solver="lbfgs"
)

logit_clf = Pipeline(steps=[
    ("prep", preprocess),
    ("model", logit)
])

# 학습
logit_clf.fit(X_tr, y_tr)

# valid AUC
val_pred_proba = logit_clf.predict_proba(X_val)[:, 1]
auc_logit = roc_auc_score(y_val, val_pred_proba)
print(f"[Logistic Regression] Valid AUC = {auc_logit:.6f}")

# test 예측 확률
test_pred_logit = logit_clf.predict_proba(test)[:, 1]

### 2) CatBoost (AUC)

In [ ]:
!pip install catboost

In [ ]:
from catboost import Pool, CatBoostClassifier
from sklearn.metrics import roc_auc_score

# 1) CatBoost용 범주형 컬럼(문자형) 자동 탐지
cat_cols = X_tr.select_dtypes(include=["object", "category", "bool", "string"]).columns.tolist()
print("cat cols:", len(cat_cols))

# 2) 범주형 컬럼 문자열 강제 변환
def to_str_cols(df, cols):
    df = df.copy()
    for c in cols:
        df[c] = df[c].astype("string").fillna("NA")
    return df

X_tr_cb  = to_str_cols(X_tr, cat_cols)
X_val_cb = to_str_cols(X_val, cat_cols)
X_test_cb = to_str_cols(test, cat_cols)

# 3) Pool 만들 때 cat_features를 '컬럼명'으로 지정 (인덱스 꼬임 방지)
train_pool = Pool(X_tr_cb, y_tr, cat_features=cat_cols)
valid_pool = Pool(X_val_cb, y_val, cat_features=cat_cols)
test_pool  = Pool(X_test_cb, cat_features=cat_cols)

# 4) 모델 학습
cb_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="AUC",
    iterations=3000,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    early_stopping_rounds=200,
    verbose=200
)

cb_model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

# valid AUC
val_pred_cb = cb_model.predict_proba(valid_pool)[:, 1]
auc_cb = roc_auc_score(y_val, val_pred_cb)
print(f"[CatBoost] Valid AUC = {auc_cb:.6f}")

# test 예측
test_pred_cb = cb_model.predict_proba(test_pool)[:, 1]

In [ ]:
import pandas as pd
import numpy as np

# 너 데이터에 id 컬럼이 있으면 여기에 적어줘. 없으면 None
ID_COL = "id"   # 예: "id" / "user_id" / "msno" 등. 없으면 None으로 바꿔.

# 제출 컬럼명(대회에서 요구하는 예측 컬럼명)
PRED_COL = "target"   # 대회가 'target' 요구하면 그대로. 아니면 'prediction'/'label' 등으로 수정.

if ID_COL is not None and ID_COL in test.columns:
    submission = pd.DataFrame({
        ID_COL: test[ID_COL].values,
        PRED_COL: test_pred_cb
    })
else:
    submission = pd.DataFrame({
        PRED_COL: test_pred_cb
    })

submission.to_csv("submission_catboost.csv", index=False)
submission.head()

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/submission_catboost.csv"
submission.to_csv(SAVE_PATH, index=False)
print("Drive 저장 완료:", SAVE_PATH)

In [ ]:
print(len(submission), len(test))